# Pig Posture Recognition - V4 Training

**Verbesserungen gegenueber V3:**
1. **Multi-Architektur Training** - DINOv2 + ConvNeXt fuer diverses Ensemble
2. **Focal Loss** fuer Minority-Klassen (optional)
3. **Class-balanced Pseudo-Labels** Integration
4. **Erweiterte Folds** - optional auch auf grossen Train-Kameras (mehr Diversitaet)
5. **Longer Training** mit mehr Patience fuer DINOv2

## Configuration

In [2]:
TAG = "T2"   # "T1" oder "T2"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

_candidates = [
    'multiview_pig_posture_recognition',
    './multiview_pig_posture_recognition',
    '/datasets/multi-view-pig-posture-recognition',
    '/multi-view-pig-posture-recognition',
]
DATA_ROOT = None
for _p in _candidates:
    if os.path.isdir(_p):
        DATA_ROOT = _p
        break
assert DATA_ROOT is not None, "Datenverzeichnis nicht gefunden!"
print(f"DATA_ROOT = {os.path.abspath(DATA_ROOT)}")

if TAG == "T1":
    CSV_PATH = f"{DATA_ROOT}/train1.csv"
    IMG_DIR  = f"{DATA_ROOT}/train1_images"
else:
    CSV_PATH = f"{DATA_ROOT}/train2.csv"
    IMG_DIR  = f"{DATA_ROOT}/train2_images"

OUTPUT_DIR = f"runs/v4_{TAG.lower()}"

# --- Architekturen: Liste, wird sequentiell trainiert ---
# Jede Architektur produziert ihre eigenen Checkpoints.
# Das Inference-Notebook mittelt ueber alle automatisch.
ARCH_LIST = [
    {
        "name": "vit_base_patch14_dinov2.lvd142m",
        "img_size": 392,        # muss durch 14 teilbar sein
        "batch_size": 64,
        "lr": 4e-4,
        "lr_backbone_mult": 0.1,
        "epochs": 25,
        "patience": 8,
        "prefix": "dinov2",
    },
    {
        "name": "convnext_base.fb_in22k_ft_in1k",
        "img_size": 384,
        "batch_size": 96,
        "lr": 5e-4,
        "lr_backbone_mult": 0.1,
        "epochs": 25,
        "patience": 8,
        "prefix": "convnext",
    },
]

# --- Training Hyperparams (global) ---
WARMUP_EPOCHS     = 3
LABEL_SMOOTH      = 0.05
MIXUP_ALPHA       = 0.1
PAD_RATIO         = 0.1
NUM_WORKERS       = 8
SEED              = 42
NUM_CLASSES       = 5

CLASS_NAMES = ["Lateral_lying_left", "Lateral_lying_right",
               "Sitting", "Standing", "Sternal_lying"]

# --- Loss-Typ ---
# "ce"    = CrossEntropy mit Klassen-Gewichten (wie V3)
# "focal" = Focal Loss (besser fuer Minority-Klassen)
LOSS_TYPE    = "focal"
FOCAL_GAMMA  = 2.0    # Hoeher = mehr Fokus auf schwere Beispiele

# --- Test-Kameras (aus Analyse) ---
TEST_CAMERAS = ["pen1_tur_cam1", "pen2_orb_cam2", "pen2_tur_cam2"]

# --- Validation-Strategie ---
# "test_only"  = CLO nur auf Test-Kameras (3 Folds, V3-Verhalten)
# "all_extra"  = CLO auf Test-Kameras + 2 groesste Train-Kameras (mehr Diversitaet)
VALIDATION_STRATEGY = "all_extra"
EXTRA_CLO_CAMS      = 2   # Top-N groesste Train-Kameras zusaetzlich

# --- Pseudo-Labeling ---
USE_PSEUDO_LABELS = False
PSEUDO_CSV        = None

# --- Fine-Tuning ---
PRETRAINED_CKPT   = None

print(f"Tag: {TAG}  |  Architekturen: {len(ARCH_LIST)}")
for a in ARCH_LIST:
    print(f"  - {a['prefix']}: {a['name']} @ {a['img_size']}px")
print(f"Validation: {VALIDATION_STRATEGY}  |  Loss: {LOSS_TYPE}  |  Output: {OUTPUT_DIR}")


DATA_ROOT = /datasets/multi-view-pig-posture-recognition
Tag: T2  |  Architekturen: 2
  - dinov2: vit_base_patch14_dinov2.lvd142m @ 392px
  - convnext: convnext_base.fb_in22k_ft_in1k @ 384px
Validation: all_extra  |  Loss: focal  |  Output: runs/v4_t2


## Imports

In [3]:
import os, ast, random, re
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import torchvision.transforms.functional as TFn
import timm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPUs: {torch.cuda.device_count()}")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)


<jemalloc>: Unsupported system page size


Device: cuda
GPU: Tesla V100-SXM2-32GB
GPUs: 4


## Daten laden

In [4]:
df = pd.read_csv(CSV_PATH)

def extract_camera(image_id):
    m = re.match(r"(pen\d+_\w+_cam\d+)", image_id)
    return m.group(1) if m else "unknown"

df["camera"] = df["image_id"].apply(extract_camera)
df["img_dir"] = IMG_DIR

if USE_PSEUDO_LABELS and PSEUDO_CSV and os.path.exists(PSEUDO_CSV):
    pseudo_df = pd.read_csv(PSEUDO_CSV)
    keep_cols = [c for c in pseudo_df.columns if c in ["row_id","image_id","width","height","bbox","class_id"]]
    pseudo_df = pseudo_df[keep_cols]
    pseudo_df["camera"] = pseudo_df["image_id"].apply(extract_camera)
    pseudo_df["img_dir"] = os.path.join(DATA_ROOT, "test_images")
    df = pd.concat([df, pseudo_df], ignore_index=True)
    print(f"Pseudo-Labels: {len(pseudo_df)} hinzugefuegt -> Gesamt: {len(df)}")

print(f"Instanzen: {len(df)}  |  Bilder: {df['image_id'].nunique()}")
print(f"Kameras ({df['camera'].nunique()}):")
for cam in sorted(df["camera"].unique()):
    cnt = (df["camera"] == cam).sum()
    print(f"  {cam:<25} {cnt:>5} ({100*cnt/len(df):.1f}%)")
print(f"Klassen:")
for c in range(NUM_CLASSES):
    cnt = (df["class_id"] == c).sum()
    print(f"  {c} - {CLASS_NAMES[c]:<22} {cnt:>5} ({100*cnt/len(df):.1f}%)")


Instanzen: 23450  |  Bilder: 3150
Kameras (8):
  pen1_orb_cam1               722 (3.1%)
  pen1_orb_cam2              1488 (6.3%)
  pen1_tur_cam1               200 (0.9%)
  pen1_tur_cam2              8194 (34.9%)
  pen2_orb_cam1              2817 (12.0%)
  pen2_orb_cam2               120 (0.5%)
  pen2_tur_cam1              9713 (41.4%)
  pen2_tur_cam2               196 (0.8%)
Klassen:
  0 - Lateral_lying_left      3083 (13.1%)
  1 - Lateral_lying_right     3435 (14.6%)
  2 - Sitting                  695 (3.0%)
  3 - Standing                9928 (42.3%)
  4 - Sternal_lying           6309 (26.9%)


## Dataset mit Label-aware Flip

In [5]:
class PigPostureDataset(Dataset):
    def __init__(self, df, transform=None, pad_ratio=0.25, is_train=False, hflip_prob=0.5):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.pad_ratio = pad_ratio
        self.is_train = is_train
        self.hflip_prob = hflip_prob

    def __len__(self):
        return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(row["img_dir"], row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        label = int(row["class_id"])

        if self.is_train and random.random() < self.hflip_prob:
            crop = TFn.hflip(crop)
            if label == 0: label = 1
            elif label == 1: label = 0

        if self.transform:
            crop = self.transform(crop)
        return crop, label


## Augmentierungen

In [6]:
class CameraSimTransform:
    def __init__(self, p=0.4):
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        if random.random() < 0.4:
            w, h = img.size
            scale = random.uniform(0.3, 0.7)
            small = img.resize((max(16, int(w*scale)), max(16, int(h*scale))), Image.BILINEAR)
            img = small.resize((w, h), Image.BILINEAR)
        if random.random() < 0.2:
            quality = random.randint(25, 65)
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=quality)
            buffer.seek(0)
            img = Image.open(buffer).convert("RGB")
        if random.random() < 0.15:
            arr = np.array(img, dtype=np.float32)
            noise = np.random.normal(0, random.uniform(5, 15), arr.shape)
            arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr)
        return img


class PerspectiveJitter:
    def __init__(self, distortion_scale=0.10, p=0.25):
        self.distortion_scale = distortion_scale
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            d = self.distortion_scale
            w, h = img.size
            return TFn.perspective(
                img,
                startpoints=[[0,0],[w,0],[w,h],[0,h]],
                endpoints=[
                    [int(random.uniform(0, w*d)), int(random.uniform(0, h*d))],
                    [int(w - random.uniform(0, w*d)), int(random.uniform(0, h*d))],
                    [int(w - random.uniform(0, w*d)), int(h - random.uniform(0, h*d))],
                    [int(random.uniform(0, w*d)), int(h - random.uniform(0, h*d))],
                ],
                fill=0
            )
        return img


def get_train_transform(size):
    return T.Compose([
        CameraSimTransform(p=0.4),
        PerspectiveJitter(distortion_scale=0.10, p=0.25),
        T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomRotation(degrees=12),
        T.RandomAffine(degrees=0, scale=(0.85, 1.15), translate=(0.05, 0.05)),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.RandomGrayscale(p=0.08),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        T.RandomErasing(p=0.20, scale=(0.02, 0.12)),
    ])


def get_val_transform(size):
    return T.Compose([
        T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

print("Augmentierungen definiert.")


Augmentierungen definiert.


## Focal Loss

Senkt den Beitrag einfacher Beispiele, fokussiert auf schwere (Minority-Klassen, Verwechslungen).
`FL(p) = -alpha * (1-p)^gamma * log(p)` mit Klassen-Gewichten als alpha.

In [7]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             label_smoothing=self.label_smoothing, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


def build_criterion(loss_type, class_weights, gamma=2.0, label_smooth=0.05):
    if loss_type == "focal":
        return FocalLoss(weight=class_weights, gamma=gamma, label_smoothing=label_smooth)
    return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smooth)


## Training Helpers

In [8]:
def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def build_optimizer(model, lr, lr_backbone_mult, weight_decay=1e-2):
    head_params, backbone_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        clean = name.replace("module.", "")
        if clean.startswith("head") or clean.startswith("fc") or clean.startswith("classifier"):
            head_params.append(param)
        else:
            backbone_params.append(param)
    print(f"  Optimizer: {len(backbone_params)} backbone (LR={lr*lr_backbone_mult:.1e}), "
          f"{len(head_params)} head (LR={lr:.1e})")
    return optim.AdamW([
        {"params": backbone_params, "lr": lr * lr_backbone_mult},
        {"params": head_params,     "lr": lr},
    ], weight_decay=weight_decay)


def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    optimizer.zero_grad()
    loss_sum, n = 0.0, 0
    preds, labels_all = [], []

    for imgs, labels in tqdm(loader, desc="  Train", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs_mix, y_a, y_b, lam = mixup_data(imgs, labels, alpha=MIXUP_ALPHA)
        with autocast():
            logits = model(imgs_mix)
            loss = mixup_loss(criterion, logits, y_a, y_b, lam)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        loss_sum += loss.item() * imgs.size(0)
        n += imgs.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(y_a.cpu().numpy())

    return loss_sum / max(n, 1), f1_score(labels_all, preds, average="macro", zero_division=0)


@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    loss_sum, n = 0.0, 0
    preds, labels_all = [], []
    for imgs, labels in tqdm(loader, desc="  Val  ", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        loss_sum += loss.item() * imgs.size(0)
        n += imgs.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    return (loss_sum / max(n, 1),
            f1_score(labels_all, preds, average="macro", zero_division=0),
            preds, labels_all)


## Folds aufbauen + Training

**Fold-Strategie:**
- `test_only`: CLO nur auf Test-Kameras (V3-Verhalten, 3 Folds in T2)
- `all_extra`: CLO auf Test-Kameras + Top-N groesste Train-Kameras (diverseres Ensemble)

In [9]:
df_train = df.copy()

available_cams = set(df_train["camera"].unique())
test_cams_in_data = sorted([c for c in TEST_CAMERAS if c in available_cams])

# Basis: Test-Kameras (falls im Datensatz vorhanden)
cams_for_clo = list(test_cams_in_data)

if len(cams_for_clo) == 0:
    # T1-Modus: keine Test-Kameras -> alle verfuegbaren
    print(f"T1-Modus: keine Test-Kameras im Datensatz, CLO auf alle {len(available_cams)} Kameras")
    cams_for_clo = sorted(available_cams)
else:
    print(f"Test-Kameras im Datensatz: {cams_for_clo}")
    if VALIDATION_STRATEGY == "all_extra" and EXTRA_CLO_CAMS > 0:
        # Top-N groesste Train-Kameras zusaetzlich (ohne Test-Kameras)
        train_cam_sizes = [(c, (df_train["camera"] == c).sum())
                           for c in available_cams if c not in test_cams_in_data]
        train_cam_sizes.sort(key=lambda x: -x[1])
        extra = [c for c, _ in train_cam_sizes[:EXTRA_CLO_CAMS]]
        cams_for_clo = cams_for_clo + extra
        print(f"  + Extra Train-Kameras: {extra}")

splits = []
fold_cameras = []
for cam in cams_for_clo:
    val_mask = df_train["camera"] == cam
    val_idx = df_train.index[val_mask].values
    train_idx = df_train.index[~val_mask].values
    if len(val_idx) > 0:
        splits.append((train_idx, val_idx))
        fold_cameras.append(cam)

n_folds = len(splits)
print(f"\n{n_folds} Folds:")
for i, cam in enumerate(fold_cameras):
    cnt = (df_train["camera"] == cam).sum()
    print(f"  Fold {i+1}: Val = {cam} ({cnt} Instanzen) | Train = {len(df_train) - cnt}")


Test-Kameras im Datensatz: ['pen1_tur_cam1', 'pen2_orb_cam2', 'pen2_tur_cam2']
  + Extra Train-Kameras: ['pen2_tur_cam1', 'pen1_tur_cam2']

5 Folds:
  Fold 1: Val = pen1_tur_cam1 (200 Instanzen) | Train = 23250
  Fold 2: Val = pen2_orb_cam2 (120 Instanzen) | Train = 23330
  Fold 3: Val = pen2_tur_cam2 (196 Instanzen) | Train = 23254
  Fold 4: Val = pen2_tur_cam1 (9713 Instanzen) | Train = 13737
  Fold 5: Val = pen1_tur_cam2 (8194 Instanzen) | Train = 15256


In [10]:
all_results = {}

for arch_idx, arch in enumerate(ARCH_LIST):
    print(f"\n{'#'*60}")
    print(f"  ARCHITEKTUR {arch_idx+1}/{len(ARCH_LIST)}: {arch['prefix']} ({arch['name']})")
    print(f"{'#'*60}")

    IMG_SIZE_ARCH  = arch["img_size"]
    BATCH_SIZE     = arch["batch_size"]
    LR             = arch["lr"]
    LR_BB_MULT     = arch["lr_backbone_mult"]
    EPOCHS_ARCH    = arch["epochs"]
    PATIENCE       = arch["patience"]
    PREFIX         = arch["prefix"]

    arch_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        print(f"\n{'='*60}")
        print(f"  [{PREFIX}] FOLD {fold_idx + 1} / {n_folds}  (Val={fold_cameras[fold_idx]})")
        print(f"{'='*60}")

        fold_train = df_train.iloc[train_idx].reset_index(drop=True)
        fold_val   = df_train.iloc[val_idx].reset_index(drop=True)

        print(f"  Train: {len(fold_train)} | Val: {len(fold_val)}")

        ckpt_path = os.path.join(OUTPUT_DIR, f"best_{PREFIX}_fold_{fold_idx+1}.pth")
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location="cpu")
            print(f"  Checkpoint existiert (val_f1={ckpt.get('val_f1',0):.4f}), skip")
            arch_results.append(ckpt.get("val_f1", 0))
            continue

        train_ds = PigPostureDataset(fold_train, transform=get_train_transform(IMG_SIZE_ARCH),
                                     pad_ratio=PAD_RATIO, is_train=True, hflip_prob=0.5)
        val_ds   = PigPostureDataset(fold_val, transform=get_val_transform(IMG_SIZE_ARCH),
                                     pad_ratio=PAD_RATIO, is_train=False)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=True)

        # --- Modell (img_size nur bei ViT-Familien setzen) ---
        try:
            model = timm.create_model(arch["name"], pretrained=True,
                                      num_classes=NUM_CLASSES, img_size=IMG_SIZE_ARCH)
        except TypeError:
            # ConvNeXt akzeptiert kein img_size-Argument
            model = timm.create_model(arch["name"], pretrained=True, num_classes=NUM_CLASSES)

        if PRETRAINED_CKPT and os.path.exists(PRETRAINED_CKPT):
            ck = torch.load(PRETRAINED_CKPT, map_location="cpu")
            model.load_state_dict(ck["model"], strict=False)

        model = model.to(DEVICE)
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)

        params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"  Modell: {arch['name']} ({params:.1f}M)")

        # --- Loss ---
        counts = Counter(fold_train["class_id"].tolist())
        weights = torch.tensor(
            [len(fold_train) / (NUM_CLASSES * max(counts.get(c, 1), 1))
             for c in range(NUM_CLASSES)], dtype=torch.float32
        ).to(DEVICE)
        criterion = build_criterion(LOSS_TYPE, weights, gamma=FOCAL_GAMMA, label_smooth=LABEL_SMOOTH)
        print(f"  Loss: {LOSS_TYPE}  |  Klassen-Gewichte: {[f'{w:.2f}' for w in weights.cpu().tolist()]}")

        # --- Optimizer + Scheduler ---
        optimizer = build_optimizer(model, LR, LR_BB_MULT)
        warmup = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS)
        cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS_ARCH - WARMUP_EPOCHS, eta_min=1e-7)
        scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])
        scaler = GradScaler()

        best_val_f1 = 0.0
        patience_counter = 0

        for epoch in range(1, EPOCHS_ARCH + 1):
            train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, scaler, criterion)
            val_loss, val_f1, val_preds, val_labels = validate_epoch(model, val_loader, criterion)
            scheduler.step()

            improved = val_f1 > best_val_f1
            mark = "*" if improved else " "
            phase = "warmup" if epoch <= WARMUP_EPOCHS else "cosine"
            print(f"  {mark} Epoch {epoch:02d}/{EPOCHS_ARCH} [{phase}] | "
                  f"Train L={train_loss:.4f} F1={train_f1:.4f} | "
                  f"Val L={val_loss:.4f} F1={val_f1:.4f}"
                  f"{' <- BEST' if improved else ''}")

            if improved:
                best_val_f1 = val_f1
                patience_counter = 0
                state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
                torch.save({
                    "epoch": epoch, "model": state,
                    "val_f1": val_f1, "model_name": arch["name"], "arch_prefix": PREFIX,
                    "tag": TAG, "img_size": IMG_SIZE_ARCH, "pad_ratio": PAD_RATIO,
                    "val_camera": fold_cameras[fold_idx],
                }, ckpt_path)
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"  Early Stopping nach {PATIENCE} Epochen")
                    break

        print(f"\n  Klassifikation Fold {fold_idx+1}:")
        for c in range(NUM_CLASSES):
            mask = np.array(val_labels) == c
            if mask.sum() > 0:
                correct = (np.array(val_preds)[mask] == c).sum()
                print(f"    {CLASS_NAMES[c]:<22} {correct}/{mask.sum()} ({100*correct/max(mask.sum(),1):.0f}%)")

        arch_results.append(best_val_f1)
        print(f"  Best Val F1: {best_val_f1:.4f}")

        del model, optimizer, scheduler, scaler
        torch.cuda.empty_cache()

    all_results[PREFIX] = arch_results
    mean_f1 = np.mean(arch_results)
    print(f"\n  [{PREFIX}] Durchschnitt ueber {len(arch_results)} Folds: {mean_f1:.4f}")

print(f"\n{'#'*60}")
print(f"  GESAMT-ERGEBNIS")
print(f"{'#'*60}")
for prefix, results in all_results.items():
    print(f"  {prefix:<12} | Mean F1: {np.mean(results):.4f} (+/- {np.std(results):.4f}) | Folds: {results}")



############################################################
  ARCHITEKTUR 1/2: dinov2 (vit_base_patch14_dinov2.lvd142m)
############################################################

  [dinov2] FOLD 1 / 5  (Val=pen1_tur_cam1)
  Train: 23250 | Val: 200
  Checkpoint existiert (val_f1=0.8167), skip

  [dinov2] FOLD 2 / 5  (Val=pen2_orb_cam2)
  Train: 23330 | Val: 120
  Checkpoint existiert (val_f1=0.8685), skip

  [dinov2] FOLD 3 / 5  (Val=pen2_tur_cam2)
  Train: 23254 | Val: 196
  Checkpoint existiert (val_f1=0.8342), skip

  [dinov2] FOLD 4 / 5  (Val=pen2_tur_cam1)
  Train: 13737 | Val: 9713
  Checkpoint existiert (val_f1=0.7845), skip

  [dinov2] FOLD 5 / 5  (Val=pen1_tur_cam2)
  Train: 15256 | Val: 8194
  Checkpoint existiert (val_f1=0.6575), skip

  [dinov2] Durchschnitt ueber 5 Folds: 0.7923

############################################################
  ARCHITEKTUR 2/2: convnext (convnext_base.fb_in22k_ft_in1k)
############################################################

  [convn